In [1]:
import os
import math
import pickle
import traceback
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from collections import deque

from qcqp_solversdg import QCQPSolver


# =========================================================
# CONFIG
# =========================================================

BASE_DIR = os.path.join(r"C:\Users\suuser\Desktop\minlplib", "paperexperiments", "synthetic_prime_minimal_revision")
PKL_PATH = os.path.join(BASE_DIR, "instances_mixed_8016_candidate.pkl")
ROOT_CSV_PATH = os.path.join(BASE_DIR, "instances_mixed_8016_root_results.csv")
# Used only if BUDGET_MODE == "ratio" (fixed mode never reads it).
RATIO_TABLE_PATH = r"root_experiment_outputs\offline_ratio_table_by_5_11da.csv"
# Optional cap on valid root instances; None = all available valid rows.
MAX_INSTANCES = None
# Only instances with successful LP and SDP_PRIME root solves enter B&B.
REQUIRE_VALID_ROOT = True

# Dimension tag — used for organized output paths
DIM_TAG      = "synthetic_prime_8016_fixed100_v1"
DATASET_NAME = f"tree_run_{DIM_TAG}"

# Output directory organized by dimension
SAVE_DIR = os.path.join(BASE_DIR, "bb_separate_trees", DIM_TAG)

START_INDEX = 0

USE_BLOCKS      = True
BLOCK_TOL       = 1e-8
VERBOSE_BLOCKS  = False
PRUNE_MATRICES  = False
MIN_BLOCK_EDGES = 0

BRANCH_RULE       = "sequential"
SPLIT_RULE        = "midpoint"
SPLIT_RULE_PARAMS = None

MAX_DEPTH = 999
MIN_WIDTH = 1e-10

# NEW: validate the fixed feasible point x = 0 and prune by its objective value.
ZERO_FEAS_TOL = 1e-8
PRUNE_TOL     = 1e-8

# How many nodes to buffer before flushing to the LIVE csv.
# Lower = safer against crashes, higher = fewer disk writes.
FLUSH_EVERY_NODES = 5

# Convert LIVE.csv to parquet after experiment finishes.
# CSV is kept during the run for crash safety, converted at the end.
CONVERT_TO_PARQUET = True
DELETE_CSV_AFTER_CONVERT = False  # preserve CSV for auditing

# =========================================================
# BUDGET MODE — switch between "fixed" and "ratio"
# =========================================================
BUDGET_MODE = "fixed"   # "fixed" or "ratio"

# Used when BUDGET_MODE = "fixed"
FIXED_NODES_LP  = 100
FIXED_NODES_SDP = 100

# Used when BUDGET_MODE = "ratio"
LP_DEPTH_REFERENCE    = 10
RATIO_COL             = "ratio_median"
FALLBACK_COL          = "global_ratio_fallback"
MANUAL_FALLBACK_RATIO = None


# =========================================================
# HELPERS
# =========================================================

# Backward-compatible class name for PKLs generated in a Jupyter __main__ module.
# The original instance attributes are restored from pickle; no re-generation occurs.
class QCQPInstance:
    pass


def load_valid_root_ids(csv_path: str) -> set:
    """Only IDs with usable, optimal LP and SDP_PRIME root results."""
    if not os.path.isfile(csv_path):
        raise FileNotFoundError(f"Root CSV not found: {csv_path}")
    root = pd.read_csv(csv_path)
    needed = {"id", "lp_status", "sdp_prime_status", "lp_obj", "sdp_prime_obj"}
    if not needed.issubset(root.columns):
        raise ValueError(f"Root CSV missing required columns: {sorted(needed - set(root.columns))}")
    if root["id"].isna().any() or root["id"].duplicated().any():
        raise ValueError("Root CSV has missing or duplicate IDs. Inspect it before running B&B.")
    lp_good = root["lp_status"].astype(str).str.strip().str.lower().eq("optimal")
    sp_good = root["sdp_prime_status"].astype(str).str.strip().str.lower().eq("optimal")
    lp_finite = np.isfinite(pd.to_numeric(root["lp_obj"], errors="coerce"))
    sp_finite = np.isfinite(pd.to_numeric(root["sdp_prime_obj"], errors="coerce"))
    good = root.loc[lp_good & sp_good & lp_finite & sp_finite, "id"]
    ids = set(good.astype(int).tolist())
    print(f"Root CSV rows: {len(root)} | valid LP + SDP_PRIME: {len(ids)} | excluded: {len(root) - len(ids)}")
    return ids


def load_instances_from_pickle(pkl_path: str) -> List[Any]:
    with open(pkl_path, "rb") as f:
        instances = pickle.load(f)
    return instances


def convert_csv_to_parquet(csv_path: str, delete_csv: bool = False) -> str:
    """
    Convert LIVE.csv to LIVE.parquet after experiment finishes.
    CSV is written during run for crash safety, converted at end.
    """
    csv_path     = os.path.abspath(csv_path)
    parquet_path = csv_path.replace(".csv", ".parquet")
    print(f"\nConverting {os.path.basename(csv_path)} → {os.path.basename(parquet_path)} ...", flush=True)
    df = pd.read_csv(csv_path)
    df.to_parquet(parquet_path, index=False)
    size_kb = os.path.getsize(parquet_path) // 1024
    print(f"Saved {parquet_path}  ({size_kb:,} KB)", flush=True)
    if delete_csv:
        os.remove(csv_path)
        print(f"Deleted {os.path.basename(csv_path)}", flush=True)
    else:
        print(f"CSV kept at {csv_path} (set DELETE_CSV_AFTER_CONVERT=True to remove)", flush=True)
    return parquet_path


def _has_finite_bounds(l, u) -> bool:
    if l is None or u is None:
        return False
    l = np.asarray(l, dtype=float)
    u = np.asarray(u, dtype=float)
    if l.size == 0 or u.size == 0:
        return False
    if l.shape != u.shape:
        return False
    if not np.all(np.isfinite(l)) or not np.all(np.isfinite(u)):
        return False
    return True


def _bounds_valid(l, u) -> bool:
    if not _has_finite_bounds(l, u):
        return False
    l = np.asarray(l, dtype=float)
    u = np.asarray(u, dtype=float)
    return np.all(u > l)


def _working_bounds_for_solve(inst_d: Dict[str, Any]) -> Tuple[np.ndarray, np.ndarray, bool]:
    bounds_exist = bool(inst_d.get("bounds_exist", False))

    l = np.asarray(inst_d.get("l", []), dtype=float)
    u = np.asarray(inst_d.get("u", []), dtype=float)

    if bounds_exist and _bounds_valid(l, u):
        return l.copy(), u.copy(), True

    l_art = np.asarray(inst_d.get("l_art", []), dtype=float)
    u_art = np.asarray(inst_d.get("u_art", []), dtype=float)

    if _bounds_valid(l_art, u_art):
        return l_art.copy(), u_art.copy(), False

    raise ValueError("No valid working bounds found.")


def _extract_instance_data(inst: Any) -> Dict[str, Any]:
    inst_d = inst.__dict__ if hasattr(inst, "__dict__") else inst

    A_list = inst_d["A_list"]
    b_list = inst_d["b_list"]
    c_list = inst_d["c_list"]

    l_work, u_work, original_bounds_exist = _working_bounds_for_solve(inst_d)

    instance_id = inst_d.get("instance_id", inst_d.get("id", None))

    n        = len(l_work)
    m_total  = len(A_list)
    m_constr = m_total - 1

    return {
        "A_list": A_list, "b_list": b_list, "c_list": c_list,
        "l_work": l_work, "u_work": u_work,
        "n": n, "m_total": m_total, "m_constr": m_constr,
        "instance_id": instance_id,
        "original_bounds_exist": original_bounds_exist,
    }


def validate_zero_incumbent(data: Dict[str, Any], tol: float = 1e-8) -> Dict[str, Any]:
    """
    Validate that x = 0 is feasible for the ORIGINAL generated QCQP.

    For the synthetic generator, every constraint constant c_k (k >= 1)
    is non-positive. The origin should therefore be feasible, but we check
    the original constraints and selected working bounds explicitly before
    using f(0) = c_0 as a fixed incumbent upper bound.
    """
    x_zero = np.zeros(data["n"], dtype=float)
    l_work = np.asarray(data["l_work"], dtype=float)
    u_work = np.asarray(data["u_work"], dtype=float)

    bounds_ok = bool(
        np.all(x_zero >= l_work - tol)
        and np.all(x_zero <= u_work + tol)
    )

    constraint_values = np.array([
        float(
            x_zero @ np.asarray(data["A_list"][k], dtype=float) @ x_zero
            + 2.0 * np.asarray(data["b_list"][k], dtype=float) @ x_zero
            + float(data["c_list"][k])
        )
        for k in range(1, len(data["A_list"]))
    ], dtype=float)

    max_constraint_value = (
        float(np.max(constraint_values))
        if constraint_values.size > 0
        else -np.inf
    )

    feasible = bool(bounds_ok and max_constraint_value <= tol)

    objective_value = float(
        x_zero @ np.asarray(data["A_list"][0], dtype=float) @ x_zero
        + 2.0 * np.asarray(data["b_list"][0], dtype=float) @ x_zero
        + float(data["c_list"][0])
    )

    return {
        "feasible": feasible,
        "bounds_ok": bounds_ok,
        "max_constraint_value": max_constraint_value,
        "upper_bound": objective_value if feasible else None,
    }


# =========================================================
# OFFLINE RATIO HELPERS  (only used when BUDGET_MODE = "ratio")
# =========================================================

def lookup_offline_ratio_by_n_m(
    ratio_table: pd.DataFrame,
    n: int,
    m_constr: int,
    ratio_col: str = "ratio_median",
    fallback_col: str = "global_ratio_fallback",
    manual_fallback: Optional[float] = None,
) -> float:
    hit = ratio_table[(ratio_table["n"] == n) & (ratio_table["m_constr"] == m_constr)]

    if not hit.empty:
        ratio = float(hit.iloc[0][ratio_col])
        if math.isfinite(ratio) and ratio > 0:
            return ratio

    if manual_fallback is not None and math.isfinite(manual_fallback) and manual_fallback > 0:
        return float(manual_fallback)

    if fallback_col in ratio_table.columns:
        fallback = float(ratio_table[fallback_col].iloc[0])
        if math.isfinite(fallback) and fallback > 0:
            return fallback

    fallback = float(ratio_table[ratio_col].median())
    if not math.isfinite(fallback) or fallback <= 0:
        raise ValueError(f"Invalid fallback offline ratio: {fallback}")

    return fallback


def compute_node_budgets_from_offline_ratio(
    ratio_table: pd.DataFrame,
    n: int,
    m_constr: int,
    lp_depth: int = 10,
    ratio_col: str = "ratio_median",
    fallback_col: str = "global_ratio_fallback",
    manual_fallback: Optional[float] = None,
) -> Dict[str, Any]:
    if lp_depth < 1:
        raise ValueError(f"lp_depth must be >= 1, got {lp_depth}")

    ratio = lookup_offline_ratio_by_n_m(
        ratio_table=ratio_table, n=n, m_constr=m_constr,
        ratio_col=ratio_col, fallback_col=fallback_col,
        manual_fallback=manual_fallback,
    )

    lp_nodes         = 2 ** lp_depth - 1
    sdp_nodes        = max(1, int(lp_nodes / ratio))
    sdp_depth_approx = int(math.log2(sdp_nodes + 1))

    return {
        "offline_ratio": ratio,
        "lp_nodes": lp_nodes,
        "sdp_nodes": sdp_nodes,
        "sdp_depth_approx": sdp_depth_approx,
        "lp_depth_reference": lp_depth,
    }


# =========================================================
# NODE
# =========================================================

@dataclass
class Node:
    node_id:   int
    parent_id: Optional[int]
    depth:     int
    l_bounds:  np.ndarray
    u_bounds:  np.ndarray
    original_bounds_exist: bool
    branch_var:  Optional[int]   = None
    split_point: Optional[float] = None
    lp_obj:           Optional[float] = None
    lp_time:          Optional[float] = None
    lp_status:        Optional[str]   = None
    sdp_prime_obj:    Optional[float] = None
    sdp_prime_time:   Optional[float] = None
    sdp_prime_status: Optional[str]   = None
    status: str           = "pending"
    note:   Optional[str] = None

    # NEW: queued children temporarily inherit their solved parent's LB.
    inherited_lb: Optional[float] = None
    known_lb:     Optional[float] = None


# =========================================================
# RULES
# =========================================================

def select_branch_variable(node: Node, rule: str = "sequential") -> int:
    if rule == "sequential":
        return node.depth % len(node.l_bounds)
    raise ValueError(f"Unknown branching rule: {rule}")


def choose_split_point(l_val, u_val, rule="midpoint", rule_params=None):
    rule_params = {} if rule_params is None else rule_params
    if rule == "midpoint":
        return 0.5 * (l_val + u_val)
    if rule == "convex_combo":
        alpha = float(rule_params.get("alpha", 0.5))
        alpha = max(0.0, min(1.0, alpha))
        return alpha * l_val + (1.0 - alpha) * u_val
    raise ValueError(f"Unknown split rule: {rule}")


def is_branchable(node: Node, branch_var: int, min_width: float = 1e-10) -> bool:
    return (node.u_bounds[branch_var] - node.l_bounds[branch_var]) > min_width


def create_children(parent_node, branch_var, split_rule, next_node_id, split_rule_params=None):
    l = parent_node.l_bounds.copy()
    u = parent_node.u_bounds.copy()
    split = choose_split_point(l[branch_var], u[branch_var],
                               rule=split_rule, rule_params=split_rule_params)
    u_left = u.copy(); u_left[branch_var] = split
    left_child = Node(
        node_id=next_node_id, parent_id=parent_node.node_id,
        depth=parent_node.depth + 1,
        l_bounds=l.copy(), u_bounds=u_left,
        original_bounds_exist=parent_node.original_bounds_exist,
        branch_var=branch_var, split_point=split,
        inherited_lb=parent_node.known_lb,
        known_lb=parent_node.known_lb,
    )
    l_right = l.copy(); l_right[branch_var] = split
    right_child = Node(
        node_id=next_node_id + 1, parent_id=parent_node.node_id,
        depth=parent_node.depth + 1,
        l_bounds=l_right, u_bounds=u.copy(),
        original_bounds_exist=parent_node.original_bounds_exist,
        branch_var=branch_var, split_point=split,
        inherited_lb=parent_node.known_lb,
        known_lb=parent_node.known_lb,
    )
    return left_child, right_child


# =========================================================
# SOLVE ONE NODE
# =========================================================

def process_node_single_relaxation(node, A_list, b_list, c_list, relaxation,
                                   use_blocks=True, block_tol=1e-8,
                                   verbose_blocks=False, prune_matrices=False,
                                   min_block_edges=0):
    if relaxation not in {"LP", "SDP_PRIME"}:
        raise ValueError(f"Unknown relaxation: {relaxation}")

    solver = QCQPSolver(
        A_list=A_list, b_list=b_list, c_list=c_list,
        l=node.l_bounds, u=node.u_bounds,
        use_blocks=use_blocks, block_tol=block_tol,
        verbose_blocks=verbose_blocks, prune_matrices=prune_matrices,
        min_block_edges=min_block_edges,
    )

    res          = solver.solve(relaxation)
    obj          = res.get("objective")
    t            = res.get("time")
    solve_status = res.get("status")

    if relaxation == "LP":
        node.lp_obj, node.lp_time, node.lp_status = obj, t, solve_status
    else:
        node.sdp_prime_obj, node.sdp_prime_time, node.sdp_prime_status = obj, t, solve_status

    if solve_status is None:
        node.status = "error"
        node.note   = "Missing solve status"
    else:
        s = str(solve_status).lower().strip()

        # Gurobi status 4 = INF_OR_UNBD. In this LP formulation, all x
        # variables have finite bounds and every created lifted X variable
        # is bounded by its McCormick envelope. Therefore, true unboundedness
        # is impossible and status 4 is treated as infeasibility.
        if s == "gurobi_status_4":
            node.status = "infeasible"
            node.note = (
                "Gurobi INF_OR_UNBD (status 4) treated as infeasible "
                "because this node relaxation is bounded."
            )

        elif "infeas" in s:
            node.status = "infeasible"

        elif "unknown" in s:
            # MOSEK could not provide a certified usable SDP_PRIME result.
            # Save the status and stop this tree later. The complete instance
            # will be removed during post-run filtering.
            node.status = "unknown"
            node.note = f"Unresolved solver status: {solve_status}"

        elif "unbound" in s:
            # Unexpected with finite working bounds; preserve and stop.
            node.status = "unbounded"
            node.note = f"Unexpected unbounded solver status: {solve_status}"

        elif "error" in s:
            node.status = "error"
            node.note = f"Solver error status: {solve_status}"

        elif "optimal" in s:
            node.status = "processed"

        else:
            node.status = "error"
            node.note = f"Unexpected solver status: {solve_status}"

    return node


def _node_relaxation_obj(node: Node, relaxation: str) -> Optional[float]:
    if relaxation == "LP":
        return node.lp_obj

    if relaxation == "SDP_PRIME":
        return node.sdp_prime_obj

    raise ValueError(f"Unknown relaxation: {relaxation}")


# =========================================================
# FRONTIER LOWER BOUND
# =========================================================
def compute_global_frontier_lb(
    queue: deque,
    open_terminal_nodes: Dict[int, Node],
    closed_by_bound_lbs: List[float],
    shared_upper_bound: Optional[float],
) -> float:
    """
    Compute the rigorous global lower bound represented by the current tree.

    Queued children use inherited parent bounds.
    Open terminal nodes retain their solved bounds.
    Bound-pruned nodes retain their solved bounds because their regions were
    removed using the incumbent-pruning rule.
    """
    represented_lbs: List[float] = []

    active_nodes = list(queue) + list(open_terminal_nodes.values())

    for active_node in active_nodes:
        if active_node.known_lb is None:
            return np.nan

        lb = float(active_node.known_lb)

        if np.isnan(lb):
            return np.nan

        represented_lbs.append(lb)

    represented_lbs.extend(float(lb) for lb in closed_by_bound_lbs)

    if represented_lbs:
        return float(min(represented_lbs))

    # No active region and no safely retained proof bound:
    # return NaN rather than treating the feasible UB as a dual bound.
    return np.nan

# =========================================================
# TREE
# =========================================================

def run_tree_single_relaxation(
    A_list, b_list, c_list, l_root, u_root,
    original_bounds_exist, relaxation,
    max_depth=999, max_nodes=None,
    branch_rule="sequential", split_rule="midpoint", split_rule_params=None,
    use_blocks=True, block_tol=1e-8, verbose_blocks=False,
    prune_matrices=False, min_block_edges=0, min_width=1e-10,
    live_csv_path=None, flush_every_nodes=5,
    dataset_name=None, instance_index=None, instance_id=None,
    n=None, m_total=None, m_constr=None,
    budget_mode=None, lp_node_budget=None, sdp_node_budget=None,
    shared_upper_bound=None, prune_tol=1e-8,
) -> Tuple[List[Node], bool]:
    """
    BFS tree for one relaxation.

    The node budget counts solved relaxations only. Queued children do not
    consume the budget, but they remain represented in the frontier through
    the valid LBs inherited from their solved parents.

    Crash safety: buffer flushed to CSV every flush_every_nodes nodes.

    Returns:
        (all_nodes, had_unresolved_status)

    If a node solve returns unknown, unbounded, or error, the row is logged
    and this relaxation tree stops immediately. The complete QCQP instance
    should then be removed during post-run filtering.
    """
    node_counter          = 0
    processed_count       = 0
    had_unresolved_status = False

    # These two containers are only for lower-bound bookkeeping.
    # BFS ordering still uses the original queue.
    open_terminal_nodes: Dict[int, Node] = {}
    closed_by_bound_lbs: List[float] = []

    root = Node(
        node_id=node_counter, parent_id=None, depth=0,
        l_bounds=np.asarray(l_root, dtype=float),
        u_bounds=np.asarray(u_root, dtype=float),
        original_bounds_exist=original_bounds_exist,
    )
    node_counter += 1

    queue        = deque([root])
    all_nodes:   List[Node]            = []
    node_buffer: List[Dict[str, Any]] = []

    def flush_node_buffer():
        nonlocal node_buffer
        if live_csv_path is None or len(node_buffer) == 0:
            return
        df_buf        = pd.DataFrame(node_buffer)
        file_exists   = os.path.exists(live_csv_path)
        header_needed = not file_exists or os.path.getsize(live_csv_path) == 0
        df_buf.to_csv(live_csv_path, mode="a", header=header_needed, index=False)
        node_buffer = []

    def _make_row(
        node: Node,
        global_frontier_lb_after_step: float,
        queue_count_after_step: int,
    ) -> Dict[str, Any]:
        row = {
            "dataset_name": dataset_name, "relaxation": relaxation,
            "node_id": node.node_id, "parent_id": node.parent_id,
            "depth": node.depth, "branch_var": node.branch_var,
            "split_point": node.split_point,
            "original_bounds_exist": node.original_bounds_exist,
            "status": node.status, "note": node.note,
            "instance_index": instance_index, "instance_id": instance_id,
            "n": n, "m_total": m_total, "m_constr": m_constr,
            "budget_mode": budget_mode,
            "lp_node_budget": lp_node_budget,
            "sdp_node_budget": sdp_node_budget,

            # NEW audit and frontier-bookkeeping fields
            "processed_count": processed_count,
            "inherited_lb": node.inherited_lb,
            "known_lb": node.known_lb,
            "shared_upper_bound": shared_upper_bound,
            "global_frontier_lb_after_step": global_frontier_lb_after_step,
            "queue_count_after_step": queue_count_after_step,
            "open_terminal_count_after_step": len(open_terminal_nodes),
            "pruned_by_bound": node.status == "pruned_by_bound",
        }
        if relaxation == "LP":
            row["obj"] = node.lp_obj
            row["solve_time"] = node.lp_time
            row["solve_status"] = node.lp_status
        else:
            row["obj"] = node.sdp_prime_obj
            row["solve_time"] = node.sdp_prime_time
            row["solve_status"] = node.sdp_prime_status
        return row

    while queue:
        if max_nodes is not None and processed_count >= max_nodes:
            break

        node = queue.popleft()
        try:
            node = process_node_single_relaxation(
                node=node, A_list=A_list, b_list=b_list, c_list=c_list,
                relaxation=relaxation, use_blocks=use_blocks, block_tol=block_tol,
                verbose_blocks=verbose_blocks, prune_matrices=prune_matrices,
                min_block_edges=min_block_edges,
            )
        except Exception as e:
            node.status = "error"
            node.note   = f"{type(e).__name__}: {str(e)}"

        all_nodes.append(node)
        processed_count += 1

        node_obj = _node_relaxation_obj(node, relaxation)
        if node.status == "processed":
            if node_obj is None or pd.isna(node_obj):
                node.status = "error"
                node.note = "Processed node has no valid relaxation objective"
            else:
                node.known_lb = float(node_obj)

        stop_tree = False

        if node.status in {"unknown", "unbounded", "error"}:
            # Preserve the inherited parent LB for this unresolved region.
            # If the root itself is unresolved, known_lb remains None and the
            # recorded global LB becomes NaN. In either case, the complete
            # instance will be removed during post-run filtering.
            open_terminal_nodes[node.node_id] = node
            had_unresolved_status = True
            print(
                f"  [{relaxation}] Unresolved node status='{node.status}' "
                f"at node={node.node_id}, depth={node.depth} — stopping tree.",
                flush=True,
            )
            stop_tree = True

        elif node.status == "infeasible":
            # Proven empty region: safely close it.
            pass

        elif (
            shared_upper_bound is not None
            and node.known_lb is not None
            and not np.isnan(float(node.known_lb))
            and float(node.known_lb) >= float(shared_upper_bound) - float(prune_tol)
        ):
            # The node is valid, but it cannot improve the feasible incumbent x = 0.
            node.status = "pruned_by_bound"
            node.note = (
                f"LB={float(node.known_lb):.12g} >= "
                f"UB={float(shared_upper_bound):.12g} - tol={float(prune_tol):.3g}"
            )
            closed_by_bound_lbs.append(float(node.known_lb))

        elif node.depth >= max_depth:
            # Not safely closed: retain as an open terminal leaf.
            open_terminal_nodes[node.node_id] = node

        else:
            branch_var = select_branch_variable(node, rule=branch_rule)

            if not is_branchable(node, branch_var=branch_var, min_width=min_width):
                # Not safely closed: retain as an open terminal leaf.
                node.status = "pruned"
                node.note = "Not branchable: interval too small"
                open_terminal_nodes[node.node_id] = node
            else:
                left_child, right_child = create_children(
                    parent_node=node, branch_var=branch_var,
                    split_rule=split_rule, split_rule_params=split_rule_params,
                    next_node_id=node_counter,
                )
                node_counter += 2
                queue.append(left_child)
                queue.append(right_child)

        global_frontier_lb_after_step = compute_global_frontier_lb(
            queue=queue,
            open_terminal_nodes=open_terminal_nodes,
            closed_by_bound_lbs=closed_by_bound_lbs,
            shared_upper_bound=shared_upper_bound,
        )

        node_buffer.append(
            _make_row(
                node=node,
                global_frontier_lb_after_step=global_frontier_lb_after_step,
                queue_count_after_step=len(queue),
            )
        )

        if len(node_buffer) >= flush_every_nodes:
            flush_node_buffer()

        if stop_tree:
            flush_node_buffer()
            break

    flush_node_buffer()
    return all_nodes, had_unresolved_status


# =========================================================
# BUDGET RESOLVER
# =========================================================

def resolve_node_budgets(
    budget_mode, data,
    fixed_nodes_lp=1000, fixed_nodes_sdp=1000,
    ratio_table=None, lp_depth=10,
    ratio_col="ratio_median", fallback_col="global_ratio_fallback",
    manual_fallback=None,
):
    if budget_mode == "fixed":
        return fixed_nodes_lp, fixed_nodes_sdp, f"FIXED LP={fixed_nodes_lp} SDP={fixed_nodes_sdp}"
    elif budget_mode == "ratio":
        if ratio_table is None:
            raise ValueError("ratio_table must be provided when budget_mode='ratio'")
        budgets = compute_node_budgets_from_offline_ratio(
            ratio_table=ratio_table, n=data["n"], m_constr=data["m_constr"],
            lp_depth=lp_depth, ratio_col=ratio_col,
            fallback_col=fallback_col, manual_fallback=manual_fallback,
        )
        desc = (f"RATIO offline_ratio={budgets['offline_ratio']:.4f} "
                f"LP={budgets['lp_nodes']} SDP={budgets['sdp_nodes']}")
        return budgets["lp_nodes"], budgets["sdp_nodes"], desc
    else:
        raise ValueError(f"Unknown budget_mode: '{budget_mode}'. Use 'fixed' or 'ratio'.")


# =========================================================
# EXPERIMENT
# =========================================================

def run_experiment_separate_trees(
    instances, dataset_name,
    max_instances=None, max_depth=999,
    budget_mode="fixed", fixed_nodes_lp=1000, fixed_nodes_sdp=1000,
    ratio_table=None, lp_depth=10,
    ratio_col="ratio_median", fallback_col="global_ratio_fallback",
    manual_fallback_ratio=None,
    branch_rule="sequential", split_rule="midpoint", split_rule_params=None,
    save_dir="experiment_outputs_separate_trees",
    start_index=0,
    eligible_root_ids=None,
    use_blocks=True, block_tol=1e-8, verbose_blocks=False,
    prune_matrices=False, min_block_edges=0, min_width=1e-10,
    flush_every_nodes=5,
    convert_to_parquet=True, delete_csv_after_convert=False,
) -> str:
    """
    Runs LP and SDP-prime trees for each instance.
    Writes all node rows to a single LIVE.csv via append-flush (crash safe).
    Converts to parquet at the end.
    Returns path to LIVE.parquet (or LIVE.csv if convert_to_parquet=False).
    """
    os.makedirs(save_dir, exist_ok=True)

    total_available = len(instances)
    selected_instances = []  # (original index, instance), preserving PKL index in outputs
    for idx in range(start_index, total_available):
        inst = instances[idx]
        d = inst.__dict__ if hasattr(inst, "__dict__") else inst
        instance_id = int(d.get("instance_id", d.get("id", idx)))
        if eligible_root_ids is not None and instance_id not in eligible_root_ids:
            continue
        selected_instances.append((idx, inst))
        if max_instances is not None and len(selected_instances) >= max_instances:
            break

    if not selected_instances:
        raise ValueError("No eligible instances selected. Check root CSV statuses and paths.")

    live_csv_path     = os.path.join(save_dir, f"{dataset_name}_LIVE.csv")
    live_parquet_path = live_csv_path.replace(".csv", ".parquet")

    # No accidental overwrites. This script does not resume an interrupted B&B tree.
    for existing_path in [live_csv_path, live_parquet_path]:
        if os.path.exists(existing_path):
            raise FileExistsError(
                f"Existing B&B output: {existing_path}. "
                "Preserved. Choose a NEW DIM_TAG for a separate run."
            )

    print("=" * 70)
    print("SEPARATE-TREE EXPERIMENT STARTED")
    print(f"Dataset name      : {dataset_name}")
    print(f"Save dir          : {save_dir}")
    print(f"Total available   : {total_available}")
    print(f"Start index       : {start_index}")
    print(f"Actual run count  : {len(selected_instances)}")
    print(f"Depth limit       : {max_depth}")
    print(f"Budget mode       : {budget_mode.upper()}")
    if budget_mode == "fixed":
        print(f"  LP nodes        : {fixed_nodes_lp}")
        print(f"  SDP nodes       : {fixed_nodes_sdp}")
    else:
        print(f"  LP depth ref    : {lp_depth}  =>  {2**lp_depth - 1} LP nodes")
        print(f"  SDP nodes       : floor(lp_nodes / ratio)  per instance")
    print(f"Flush every nodes : {flush_every_nodes}")
    print(f"Convert to parquet: {convert_to_parquet}")
    print(f"LIVE csv          : {live_csv_path}")
    print("=" * 70)

    instances_ok         = 0
    instances_unresolved = 0
    instances_error      = 0

    for local_idx, (global_idx, inst) in enumerate(selected_instances):
        print(f"\n[{local_idx + 1}/{len(selected_instances)}] instance {global_idx}")

        try:
            data = _extract_instance_data(inst)

            # NEW: use the known feasible point x = 0 for the ORIGINAL QCQP.
            zero_info = validate_zero_incumbent(data, tol=ZERO_FEAS_TOL)
            if not zero_info["feasible"]:
                raise ValueError(
                    "x=0 is not feasible for this synthetic instance: "
                    f"bounds_ok={zero_info['bounds_ok']}, "
                    f"max_constraint_value={zero_info['max_constraint_value']}"
                )
            shared_upper_bound = float(zero_info["upper_bound"])

            if not np.isfinite(shared_upper_bound):
                raise ValueError(
                    f"Non-finite shared upper bound obtained from x=0: "
                    f"{shared_upper_bound}"
                )

            lp_nodes, sdp_nodes, budget_desc = resolve_node_budgets(
                budget_mode=budget_mode, data=data,
                fixed_nodes_lp=fixed_nodes_lp, fixed_nodes_sdp=fixed_nodes_sdp,
                ratio_table=ratio_table, lp_depth=lp_depth,
                ratio_col=ratio_col, fallback_col=fallback_col,
                manual_fallback=manual_fallback_ratio,
            )
            print(f"  size=(n={data['n']}, m={data['m_constr']}) | {budget_desc}")
            print(f"  fixed incumbent : x=0 | shared UB={shared_upper_bound:.12g}")

            tree_kwargs = dict(
                A_list=data["A_list"], b_list=data["b_list"], c_list=data["c_list"],
                l_root=data["l_work"], u_root=data["u_work"],
                original_bounds_exist=data["original_bounds_exist"],
                max_depth=max_depth,
                branch_rule=branch_rule, split_rule=split_rule,
                split_rule_params=split_rule_params,
                use_blocks=use_blocks, block_tol=block_tol,
                verbose_blocks=verbose_blocks, prune_matrices=prune_matrices,
                min_block_edges=min_block_edges, min_width=min_width,
                live_csv_path=live_csv_path,
                flush_every_nodes=flush_every_nodes,
                dataset_name=dataset_name,
                instance_index=global_idx, instance_id=data["instance_id"],
                n=data["n"], m_total=data["m_total"], m_constr=data["m_constr"],
                budget_mode=budget_mode,
                lp_node_budget=lp_nodes,
                sdp_node_budget=sdp_nodes,
                shared_upper_bound=shared_upper_bound,
                prune_tol=PRUNE_TOL,
            )

            _, lp_unresolved = run_tree_single_relaxation(
                relaxation="LP",
                max_nodes=lp_nodes,
                **tree_kwargs,
            )

            if lp_unresolved:
                instances_unresolved += 1
                print(
                    f"  instance {global_idx} marked unresolved after LP; "
                    "SDP_PRIME tree skipped. Remove the complete instance "
                    "during post-run filtering.",
                    flush=True,
                )
                continue

            _, sdp_unresolved = run_tree_single_relaxation(
                relaxation="SDP_PRIME",
                max_nodes=sdp_nodes,
                **tree_kwargs,
            )

            if sdp_unresolved:
                instances_unresolved += 1
                print(
                    f"  instance {global_idx} marked unresolved after SDP_PRIME. "
                    "Remove the complete instance during post-run filtering.",
                    flush=True,
                )
                continue

            instances_ok += 1

        except Exception as e:
            instances_error += 1
            print(f"ERROR on instance {global_idx}: {type(e).__name__}: {str(e)}")
            print(traceback.format_exc())

    print("\n" + "=" * 70)
    print("SEPARATE-TREE EXPERIMENT FINISHED")
    print(f"Instances OK         : {instances_ok}")
    print(f"Instances UNRESOLVED : {instances_unresolved}")
    print(f"Instances ERROR      : {instances_error}")
    print(f"LIVE csv          : {live_csv_path}")
    print("=" * 70)

    # Convert CSV → parquet after run finishes
    output_path = live_csv_path
    if convert_to_parquet and os.path.exists(live_csv_path):
        output_path = convert_csv_to_parquet(live_csv_path, delete_csv=delete_csv_after_convert)

    return output_path


# =========================================================
# MAIN
# =========================================================

if __name__ == "__main__":
    instances = load_instances_from_pickle(PKL_PATH)
    valid_root_ids = load_valid_root_ids(ROOT_CSV_PATH) if REQUIRE_VALID_ROOT else None
    ratio_table = pd.read_csv(RATIO_TABLE_PATH) if BUDGET_MODE == "ratio" else None

    live_path = run_experiment_separate_trees(
        instances=instances,
        dataset_name=DATASET_NAME,
        max_instances=MAX_INSTANCES,
        max_depth=MAX_DEPTH,
        budget_mode=BUDGET_MODE,
        fixed_nodes_lp=FIXED_NODES_LP,
        fixed_nodes_sdp=FIXED_NODES_SDP,
        ratio_table=ratio_table,
        lp_depth=LP_DEPTH_REFERENCE,
        ratio_col=RATIO_COL,
        fallback_col=FALLBACK_COL,
        manual_fallback_ratio=MANUAL_FALLBACK_RATIO,
        branch_rule=BRANCH_RULE,
        split_rule=SPLIT_RULE,
        split_rule_params=SPLIT_RULE_PARAMS,
        save_dir=SAVE_DIR,
        start_index=START_INDEX,
        eligible_root_ids=valid_root_ids,
        use_blocks=USE_BLOCKS,
        block_tol=BLOCK_TOL,
        verbose_blocks=VERBOSE_BLOCKS,
        prune_matrices=PRUNE_MATRICES,
        min_block_edges=MIN_BLOCK_EDGES,
        min_width=MIN_WIDTH,
        flush_every_nodes=FLUSH_EVERY_NODES,
        convert_to_parquet=CONVERT_TO_PARQUET,
        delete_csv_after_convert=DELETE_CSV_AFTER_CONVERT,
    )

    print(f"\nOutput: {live_path}")
    print("Pass this file to summarize_separate_tree_results() to get the summary.")


Root CSV rows: 6150 | valid LP + SDP_PRIME: 6149 | excluded: 1
SEPARATE-TREE EXPERIMENT STARTED
Dataset name      : tree_run_synthetic_prime_8016_fixed100_v1
Save dir          : C:\Users\suuser\Desktop\minlplib\paperexperiments\synthetic_prime_minimal_revision\bb_separate_trees\synthetic_prime_8016_fixed100_v1
Total available   : 8016
Start index       : 0
Actual run count  : 6149
Depth limit       : 999
Budget mode       : FIXED
  LP nodes        : 100
  SDP nodes       : 100
Flush every nodes : 5
Convert to parquet: True
LIVE csv          : C:\Users\suuser\Desktop\minlplib\paperexperiments\synthetic_prime_minimal_revision\bb_separate_trees\synthetic_prime_8016_fixed100_v1\tree_run_synthetic_prime_8016_fixed100_v1_LIVE.csv

[1/6149] instance 0
  size=(n=80, m=30) | FIXED LP=100 SDP=100
  fixed incumbent : x=0 | shared UB=-6.63190377369
Set parameter Username
Academic license - for non-commercial use only - expires 2027-09-30

[2/6149] instance 1
  size=(n=10, m=20) | FIXED LP=100 SDP=